# 10 — CNN rung 4, experiment 2: LR schedule (cosine annealing)

**Decision this feeds** (`RESOURCES.md`'s Chegodaev et al. 2026 finding
— "training protocol mattered more than architecture or ensembling"):
`config.LR_SCHEDULE = None` (constant LR) throughout rungs 0-3, flagged
from the start as "an experiment to try," never actually tried. This
experiment adds `torch.optim.lr_scheduler.CosineAnnealingWarmRestarts`
(`train_one_fold`'s `scheduler` argument already supports this — it's
been plumbed and unit-tested since the rung 2/3 spec, just never
exercised with a real scheduler), with Chegodaev et al.'s concrete
values: `T_0=50, T_mult=1, eta_min=1e-6`.

**Important nuance**: `T_0=50` equals `config.EPOCHS` (the training-loop
cap), and rung 3's own fold-by-fold epoch counts (`README.md`) ranged
12-30 — every fold stopped early (`config.PATIENCE=10`) well before
epoch 50. That means **the "warm restart" never actually fires** in
practice; `T_mult` is moot. What this experiment really tests is a
*single smooth cosine decay toward `eta_min` over the training run*,
not restarts. Worth knowing before reading the result as a verdict on
"warm restarts" specifically — it isn't one.

**Gate**: same as experiment 1 (`notebooks/09`) — nested-CV + paired
bootstrap against the **current validated CNN** (rung 3, `README.md`
2026-09-09: mean=0.4520, sd=0.0109), not the classical baseline. No
hyperparameter search here (Chegodaev et al.'s literal values, no
grid) — straight to a fold-0 sanity check, then the full 5×5 gate.

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers, never any per-row output.

In [ ]:
# [RUN ME] -- loads real pixel data + row-level labels. Reuses the
# shared on-disk volume cache (this experiment doesn't touch
# preprocessing, so cell 08's "reused" result should hold).
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
print(f"cache {'reused' if volume_cache.was_reused else 'rebuilt'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")

In [ ]:
# [RUN ME] (no data access itself). Same helper as notebooks 06/07/09,
# with an optional CosineAnnealingWarmRestarts in place of the constant
# LR train_one_fold already supports a `scheduler` argument for.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            use_lr_schedule=False,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    scheduler = None
    if use_lr_schedule:
        # Chegodaev et al. 2026 (RESOURCES.md): T_0 = config.EPOCHS since
        # early stopping means we never observe a restart in practice --
        # see this notebook's intro cell.
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=epochs, T_mult=1, eta_min=1e-6)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed, scheduler=scheduler,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state

In [ ]:
# [RUN ME] -- fold-0 sanity check (no hyperparameter search -- Chegodaev
# et al.'s literal values, see intro cell). Confirms the scheduler runs
# without error and reports a first, cheap outer-fold number before
# committing to the full 5x5 gate below.
batch_size, lr = 32, 2e-3  # rung 2/3's validated winner, held fixed here

outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
fold0_train_idx, fold0_test_idx = outer_folds[0]
fold0_train_uids = [uids[i] for i in fold0_train_idx]
fold0_train_labels = [labels[i] for i in fold0_train_idx]
fold0_train_family = [families[i] for i in fold0_train_idx]
fold0_test_uids = [uids[i] for i in fold0_test_idx]
fold0_test_labels = np.array([labels[i] for i in fold0_test_idx])

start = time.time()
probs, history, best_state = train_and_score_nested(
    fold0_train_uids, fold0_train_labels, fold0_train_family,
    fold0_test_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
    use_lr_schedule=True,
)
elapsed = time.time() - start
score = evaluate.log_loss_score(fold0_test_labels, probs)
print(f"fold 0 sanity check: {len(history['val_loss'])} epochs, inner-val best="
      f"{min(history['val_loss']):.4f}, outer log loss={score:.4f}, {elapsed:.1f}s "
      f"({elapsed / len(history['val_loss']):.2f}s/epoch)")
print("rung 3's fold-0/seed-42 log loss for comparison: 0.4005 (README.md)")

In [ ]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x, with the cosine LR
# schedule. Same protocol as rung 3 (notebooks/07_cnn_rung3.ipynb).
N_REPEATS = 5
oof_repeats_lrsched = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        probs, history, best_state = train_and_score_nested(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
            use_lr_schedule=True,
        )
        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"rung4_lrsched_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
              f"outer fold log loss={fold_score:.4f}")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_lrsched.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"rung4_lrsched_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_lrsched = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_lrsched])
print(f"\n{N_REPEATS}-repeat cosine-LR-schedule CNN pooled log loss: "
      f"mean={repeat_scores_lrsched.mean():.4f}, sd={repeat_scores_lrsched.std(ddof=1):.4f}")
print("current validated CNN (rung 3, README.md 2026-09-09): mean=0.4520, sd=0.0109")

In [ ]:
# [RUN ME] -- paired bootstrap: cosine-LR-schedule CNN (this
# experiment's repeat 0, seed=42) vs. the current validated CNN (rung
# 3's repeat 0, seed=42, same split -- reloaded from disk, not
# retrained). Same discipline as notebooks 07 and 09's gate cells.
CURRENT_CNN_MEAN = 0.4520  # README.md 2026-09-09, rung 3, 5-repeat mean
CURRENT_CNN_SD = 0.0109    # same, ddof=1

y_true = np.array(labels)
current_cnn_oof = np.load(config.DATA_PROCESSED / "rung3_oof_seed42.npy")
lrsched_oof_for_pairing = oof_repeats_lrsched[0]

ci_low, ci_high = evaluate.paired_bootstrap_ci(
    y_true, lrsched_oof_for_pairing, current_cnn_oof, seed=config.SEED)
ci_favors_lrsched = ci_high < 0  # delta = lrsched - current; negative favors lrsched

repeat_mean_lrsched = repeat_scores_lrsched.mean()
repeat_sd_lrsched = repeat_scores_lrsched.std(ddof=1)
noise_threshold = 2 * max(repeat_sd_lrsched, CURRENT_CNN_SD)
beats_current_by = CURRENT_CNN_MEAN - repeat_mean_lrsched

gate_passed = ci_favors_lrsched and (beats_current_by > noise_threshold)

print(f"paired bootstrap delta (cosine-LR-schedule - current CNN), 95% CI: [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"5-repeat mean={repeat_mean_lrsched:.4f}, sd={repeat_sd_lrsched:.4f}; "
      f"beats current CNN ({CURRENT_CNN_MEAN}) by {beats_current_by:+.4f} "
      f"(2x max-sd noise threshold = {noise_threshold:.4f})")
print(f"GATE {'PASSED' if gate_passed else 'NOT PASSED'}: "
      f"{'cosine LR schedule REPLACES the current CNN.' if gate_passed else 'does not beat the current CNN by more than noise -- keep the current CNN.'}")

**What we're looking for:** does a cosine-decay LR schedule (in place of
the constant LR used in rungs 0-3) beat the current validated CNN
(0.4520) by more than noise?

**What we found:** *(paste: the fold-0 sanity check number; the
5-repeat mean/sd; the paired-bootstrap 95% CI; the GATE PASSED/NOT
PASSED line)*

**Decision / next step:** *(if the gate passed: this CNN replaces the
current one in the submission blend -- re-run the blend-weight
leave-one-repeat-out check against it, since w_cnn=0.70 was tuned for
the old CNN. If not: keep the current CNN, move on to experiment 3
[augmentation, `notebooks/11_cnn_augmentation.ipynb`], and log this as a
negative result per the project's standing rule.)*